# Baseline Model — NYC Green Taxi Trip Duration

Module 1 — Mini Project 1: quick baseline before packaging into a service.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle

## 1. Load data

In [ ]:
df = pd.read_parquet("../data/green_tripdata_2023-01.parquet")
print(df.shape)
df.head()

## 2. Compute target: trip duration (minutes)

In [ ]:
df["duration"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60

# drop unrealistic trips (0 min or > 60 min) so they don't distort the model
df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
print(df.shape)
df["duration"].describe()

## 3. Simple feature engineering

In [ ]:
df["PU_DO"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)

categorical = ["PU_DO"]
numerical = ["trip_distance"]

df[categorical] = df[categorical].astype(str)

## 4. Train / validation split

In [ ]:
n = len(df)
split_idx = int(n * 0.8)

df_train = df.iloc[:split_idx]
df_val = df.iloc[split_idx:]

print(f"train: {len(df_train)}, val: {len(df_val)}")

## 5. Vectorize features

In [ ]:
dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient="records")
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient="records")
X_val = dv.transform(val_dicts)

y_train = df_train["duration"].values
y_val = df_val["duration"].values

## 6. Train the model

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

## 7. Evaluate (MAE, RMSE)

In [ ]:
y_pred = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"MAE:  {mae:.3f} minutes")
print(f"RMSE: {rmse:.3f} minutes")

## 8. Save the model (vectorizer + regressor)

In [ ]:
with open("../models/baseline.pkl", "wb") as f_out:
    pickle.dump((dv, model), f_out)

print("Model saved to models/baseline.pkl")